In [ ]:
import pandas as pd
import numpy as np

# sample_data 폴더 내의 CSV 파일 경로 지정
# 해당 데이터는 직접 크롤링 & 데이터 전처리를 통해 수집한 자료입니다.
file_path = '/content/sample_data/scentive_master_db_v10.csv'

#파일을 읽는 순간 데이터프레임으로 변환
df = pd.read_csv(file_path)
print('== 해당 파일의 차원의 형태는 : ', df.shape, '==')

== 해당 파일의 차원의 형태는 :  (2387, 23) ==


In [ ]:
#23개의 열 중 4개의 열만 사용
#4개 : 'Molecular_Weight', 'XLogP', 'Flash_Point_Celsius', 'Note_Classification'
#3개를 활용해서 Note_Classification 예측하기
use_df = df[['Molecular_Weight', 'XLogP', 'Flash_Point_Celsius', 'Note_Classification']]

print("=== 최종적으로 사용할 데이터 목록 - 중간 점검 === \n", use_df.head())

=== 최종적으로 사용할 데이터 목록 - 중간 점검 === 
    Molecular_Weight  XLogP  Flash_Point_Celsius Note_Classification
0               NaN    NaN                43.33                Base
1            223.35    4.4               224.44          Top,Middle
2               NaN    NaN                84.44                Base
3            218.33    2.4               100.00         Middle,Base
4               NaN    NaN                93.33     Top,Middle,Base


In [ ]:
print(f"===None을 삭제하기 전=== : \n{use_df['Molecular_Weight'].head()}")

===None을 삭제하기 전=== : 
0       NaN
1    223.35
2       NaN
3    218.33
4       NaN
Name: Molecular_Weight, dtype: float64


In [ ]:
#NaN가 들어있는 행 데이터 삭제
use_df = use_df.dropna()

print(f"===None을 삭제하고 나서=== : \n{use_df['Molecular_Weight'].head()}")

===None을 삭제하고 나서=== : 
1    223.35
3    218.33
5    172.26
6    178.27
8    260.40
Name: Molecular_Weight, dtype: float64


In [ ]:
#Note_Classification 중 한 가지로만 분류되어있는 향료들만 사용하기
Note_Classification = (use_df['Note_Classification'] == 'Middle') | (use_df['Note_Classification'] == 'Top') | (use_df['Note_Classification'] == 'Base')
use_df = use_df.loc[Note_Classification]

print("===== 최종적으로 사용할 데이터 목록 - 최종 =====")
print(f"최종 데이터 길이는 : {len(use_df)}\n")
print(f"{use_df}")

===== 최종적으로 사용할 데이터 목록 - 최종 =====
최종 데이터 길이는 : 728

      Molecular_Weight  XLogP  Flash_Point_Celsius Note_Classification
5               172.26    2.8                78.89                 Top
8               260.40    4.8               100.00                Base
10              174.28    3.2                48.80                 Top
11              234.38    4.2               104.44                Base
17              264.40    4.4                93.33                Base
...                ...    ...                  ...                 ...
2365            208.34    3.3               100.00                Base
2370            130.18    1.7                35.00                 Top
2377            134.18    0.8                78.89                Base
2383            204.26    3.1               110.00              Middle
2385            216.32    4.6               100.00              Middle

[728 rows x 4 columns]


In [ ]:
#knn을 위해 각각 top, middle, base에 0,1,2 숫자 부여 후 열 추가
note_dict = {'Top': 0, 'Middle': 1, 'Base': 2}
use_df['Note_dict'] = use_df['Note_Classification'].map(note_dict)

print(f"{use_df}")

      Molecular_Weight  XLogP  Flash_Point_Celsius Note_Classification  \
5               172.26    2.8                78.89                 Top   
8               260.40    4.8               100.00                Base   
10              174.28    3.2                48.80                 Top   
11              234.38    4.2               104.44                Base   
17              264.40    4.4                93.33                Base   
...                ...    ...                  ...                 ...   
2365            208.34    3.3               100.00                Base   
2370            130.18    1.7                35.00                 Top   
2377            134.18    0.8                78.89                Base   
2383            204.26    3.1               110.00              Middle   
2385            216.32    4.6               100.00              Middle   

      Note_dict  
5             0  
8             2  
10            0  
11            2  
17            2  
...

In [ ]:
#학습데이터 분할
train_size = int(0.6 * len(use_df))
train_data = use_df[:train_size]

#검증데이터 분할
validation_size = int(0.2 * len(use_df))
validation_data = use_df[train_size:train_size+validation_size]

#테스트데이터 분할
test_data = use_df[train_size+validation_size:]

print(f"학습데이터 길이는 {train_size}")
print(f"검증데이터 길이는 {validation_size}")
print(f"테스트데이터 길이는 {len(test_data)}")
print(f"총 {train_size + validation_size + len(test_data)}")

학습데이터 길이는 436
검증데이터 길이는 145
테스트데이터 길이는 147
총 728


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
X_train = train_data[['Molecular_Weight', 'XLogP', 'Flash_Point_Celsius']]
y_train = train_data.Note_dict

X_validation = validation_data[['Molecular_Weight', 'XLogP', 'Flash_Point_Celsius']]
y_validation = validation_data.Note_dict

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_validation = scaler.transform(X_validation)

#에러 고치기 - 🤖
#pandas Series에는 .values라는 게 붙어 있는데, 이걸 쓰면 pandas 껍데기를 벗기고 안의 numpy 배열만 빼낼 수 있어.
#그러면 tensor가 모양을 알아볼 수 있게 돼. y_train, y_validation 둘 다 같은 처리가 필요해.
y_train = y_train.to_numpy()
y_validation = y_validation.to_numpy()

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_validation_tensor = torch.tensor(X_validation, dtype=torch.float32)
y_validation_tensor = torch.tensor(y_validation, dtype=torch.long)

In [ ]:
#모델 생성
k = 5
knn_model = KNeighborsClassifier(n_neighbors=k)

knn_model.fit(X_train_tensor.numpy(), y_train_tensor.numpy())

KNeighborsClassifier()

In [ ]:
#모델 평가
accuracy = knn_model.score(X_validation_tensor.numpy(), y_validation_tensor.numpy())
print(f"테스트 정확도 : {accuracy}")

테스트 정확도 : 0.5103448275862069


In [ ]:
#샘플 데이터 예측
sample = X_validation_tensor[0].reshape(1,-1).numpy()

print(X_validation_tensor[0])
print(X_validation_tensor[0].reshape(1,-1))
print(X_validation_tensor[0].reshape(1,-1).numpy())

tensor([ 0.3047, -0.0102,  0.6292])
tensor([[ 0.3047, -0.0102,  0.6292]])
[[ 0.30466762 -0.01020591  0.6291602 ]]


In [ ]:
prediction = knn_model.predict(sample)

print(prediction)
print(prediction[0])

[1]
1


In [ ]:
#🤖 - 예측된 클래스 출력하기
reverse_dict = {v: k for k, v in note_dict.items()}

predicted_class = reverse_dict[prediction[0]]
print(f"예측된 클래스:\n {predicted_class}")

예측된 클래스:
 Middle


In [ ]:
#🤖 - 이번에는 test 데이터를 가지고 모델이 잘 작동하는지 확인하기
X_test = test_data[['Molecular_Weight', 'XLogP', 'Flash_Point_Celsius']]
y_test = test_data.Note_dict

X_test = scaler.transform(X_test)

#모델로 예측하기
y_pred = knn_model.predict(X_test)

#예측한 값이 맞는지 확인하기
result_df = pd.DataFrame({'정답': y_test, '예측': y_pred})
result_df['일치'] = result_df['정답'] == result_df['예측']
print(result_df, '\n')

answer = result_df['일치'].sum()
print(f"맞은 개수: {answer} / 전체 {len(result_df)}")

      정답  예측     일치
1876   0   0   True
1882   2   1  False
1884   2   1  False
1890   2   1  False
1891   1   1   True
...   ..  ..    ...
2365   2   1  False
2370   0   0   True
2377   2   0  False
2383   1   1   True
2385   1   2  False

[147 rows x 3 columns] 

맞은 개수: 72 / 전체 147


#클로드 대화 내역
https://claude.ai/share/9192c876-07bf-4bbb-82d3-c8fb12922869